In [8]:
import json
from tqdm import tqdm

In [13]:
with open('Japanese-Eroge-Voice.json') as fopen:
    rows = json.load(fopen)

In [14]:
len(rows)

217141

In [15]:
mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 217141/217141 [00:00<00:00, 2683704.46it/s]


217141

In [23]:
import faiss
import os
import numpy as np

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'Japanese-Eroge-Voice/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 217141/217141 [00:57<00:00, 3770.06it/s]


In [24]:
len(data)

217141

In [26]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [27]:
rows[0]

{'audio_filename': 'Japanese-Eroge-Voice_audio/Japanese-Eroge-Voice-ae1422b120c2d902.mp3',
 'text': '戻しときましょ。それともあんた、食べてみたいの?',
 'speaker': 'Japanese-Eroge-Voice_audio_0'}

In [28]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'Japanese-Eroge-Voice_audio/Japanese-Eroge-Voice-ae1422b120c2d902.mp3',
 'text': '戻しときましょ。それともあんた、食べてみたいの?',
 'speaker': 'Japanese-Eroge-Voice_audio_0'}

In [29]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Japanese-Eroge-Voice')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 12.77ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  99%|█████████▉| 13.7MB / 13.8MB, 34.2MB/s  
Processing Files (1 / 1): 100%|██████████| 13.8MB / 13.8MB, 23.0MB/s  
Processing Files (1 / 1): 100%|██████████| 13.8MB / 13.8MB, 17.3MB/s  
New Data Upload: 100%|██████████| 13.8MB / 13.8MB, 17.3MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.45s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/0ca1acdb74fb24a0e18ef7edb5f669994ade0851', commit_message='Upload dataset', commit_description='', oid='0ca1acdb74fb24a0e18ef7edb5f669994ade0851', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)